# Benchmarking Tabular Explainer
To benchmark `shapiq`'s `TabularExplainer` we run it against the equivalent explainers of the shap library. For comparing different approximators and imputer strategies We run both, the `shapiq` and the `shap` versions against the exact calculation of shapley values.

In [ ]:
from __future__ import annotations

import numpy as np
import shapiq

## Preparing Datasets and models
We use the `california_housing` and the `bike_sharing` datasets for the benchmark as they are comparably small datasets (so the exact shapley values can be computed in reasonable time) still they are realistic (real world) data sets.
The values are scaled to a range from -1 to 1 and split up in training and test data (80:20).
As both data sets have non-classified output, we use a regression model (scikit-learn's`DecisionTreeRegressor`) to be explained in the benchmark. We use scikit-learn's  as model.

In [ ]:
bike_tree = shapiq.games.benchmark.setup.GameBenchmarkSetup(
    dataset_name="bike_sharing", model_name="decision_tree"
)
house_tree = shapiq.games.benchmark.setup.GameBenchmarkSetup(
    dataset_name="california_housing", model_name="decision_tree"
)
bike_x_train = bike_tree.x_train
bike_x_test = bike_tree.x_test
bike_y_train = bike_tree.y_train
bike_y_test = bike_tree.y_test

house_x_train = house_tree.x_train
house_x_test = house_tree.x_test
house_y_train = house_tree.y_train
house_y_test = house_tree.y_test

##Imputation
As the performance concerning speed as well as quality of an approximator depends strongly on the used imputer, we are going to use different imputers for comparison.
On the other hand the performance of an approximator depends on the number of passes so we are going to run the test on the same amount for each approximator.


In [ ]:
sample_size = 100
# The number of samples to draw from the conditional background data for the imputation.

conditional_budget = np.arange(10, 101, 10)
# The number of coalitions to sample per each point in `data` for training the generative model.

conditional_threshold = 10
# Quantile threshold defining a neighbourhood of samples to draw `sample_size` from.

random_state = 42

## Running Shapiq's TabularExplainer

In [ ]:
kern_house = shapiq.approximator.KernelSHAP(len(house_x_train[0]))
kern_bike = shapiq.approximator.KernelSHAP(len(bike_x_train[0]))

svarm_house = shapiq.approximator.SVARM(
    n=len(house_x_train[0]), index="SV", random_state=random_state
)
svarm_bike = shapiq.approximator.SVARM(
    n=len(bike_x_train[0]), index="SV", random_state=random_state
)

perm_house = shapiq.approximator.PermutationSamplingSV(
    n=len(house_x_train[0]), random_state=random_state
)
perm_bike = shapiq.approximator.PermutationSamplingSV(
    n=len(bike_x_train[0]), random_state=random_state
)

house_tree_kernel = shapiq.TabularExplainer(
    model=house_tree.model,
    data=house_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=kern_house,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)
bike_tree_kernel = shapiq.TabularExplainer(
    model=bike_tree.model,
    data=bike_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=kern_bike,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)
house_tree_svarm = shapiq.TabularExplainer(
    model=house_tree.model,
    data=house_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=svarm_house,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)
bike_tree_svarm = shapiq.TabularExplainer(
    model=bike_tree.model,
    data=bike_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=svarm_bike,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)

house_tree_perm = shapiq.TabularExplainer(
    model=house_tree.model,
    data=house_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=perm_house,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)

bike_tree_perm = shapiq.TabularExplainer(
    model=bike_tree.model,
    data=bike_tree.x_train,
    class_index=None,
    imputer="marginal",
    approximator=perm_bike,
    index="SV",
    max_order=1,
    random_state=random_state,
    verbose=True,
    sample_size=sample_size,
    joint_marginal_distribution=False,
    normalize=False,
)

for j in range(3):
    for i in range(len(conditional_budget)):
        h_t_k_expl = house_tree_kernel.explain_function(
            x=house_tree.x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        display(h_t_k_expl)

        b_t_k_expl = bike_tree_kernel.explain_function(
            x=bike_tree.x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        display(b_t_k_expl)

        h_t_s_expl = house_tree_svarm.explain_function(
            x=house_tree.x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        display(h_t_s_expl)

        b_t_s_expl = bike_tree_svarm.explain_function(
            x=bike_tree.x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        display(b_t_s_expl)

        h_t_p_expl = house_tree_perm.explain_function(
            x=house_tree.x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        display(h_t_p_expl)

        b_t_p_expl = bike_tree_perm.explain_function(
            x=bike_tree.x_test[j], budget=conditional_budget[i], random_state=random_state
        )
        display(b_t_p_expl)

## Running Shap's Explainers